# DNA Baseline Features (Traditional)

This notebook builds **traditional (non-LM)** DNA features for the promoter vs non-promoter dataset.

**Input:** `data/processed/dna_promoter_vs_nonpromoter_len{L}_pos{N_POS}_neg{N_NEG}.csv`  
**Output (features):**
- Feature matrix saved as:
  - `data/processed/dna_features_baseline_len{L}_pos{N_POS}_neg{N_NEG}.parquet`
- A feature-only CSV (optional, smaller columns):
  - `data/processed/dna_features_baseline_len{L}_pos{N_POS}_neg{N_NEG}.csv`
- Feature schema + summary:
  - `reports/dna_baseline_features_summary.json`

## Feature set (traditional baseline)

1. **Base composition**
- A/C/G/T fractions, N fraction
- GC content, AT content

2. **Skews**
- GC skew = (G − C)/(G + C)
- AT skew = (A − T)/(A + T)

3. **CpG biology**
- CpG density (count("CG") / (L-1))
- CpG observed/expected

4. **k-mers**
- 2-mers (dinucleotide) frequencies (16)
- 3-mers (trinucleotide) frequencies (64)

5. **Sequence complexity**
- Shannon entropy over {A,C,G,T} (ignore N)
- Max homopolymer run (A/C/G/T)

6. **Simple periodicity proxy**
- fraction of AA/TT/TA dinucleotides

In [1]:
from pathlib import Path
import json
import math
import random

import numpy as np
import pandas as pd
import yaml

ROOT = Path.cwd().parents[0]
PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
CONFIGS = ROOT / "configs"

for p in [PROCESSED, REPORTS]:
    p.mkdir(parents=True, exist_ok=True)

cfg_path = CONFIGS / "config.yaml"
assert cfg_path.exists(), f"Missing config: {cfg_path}"

with open(cfg_path, "r") as f:
    cfg = yaml.safe_load(f)

SEED = int(cfg["project"]["random_seed"])
L = int(cfg["dna"]["seq_length_bp"])
N_POS = int(cfg["dna"]["n_pos"])
N_NEG = int(cfg["dna"]["n_neg"])

random.seed(SEED)
np.random.seed(SEED)

in_csv = PROCESSED / f"dna_promoter_vs_nonpromoter_len{L}_pos{N_POS}_neg{N_NEG}.csv"
assert in_csv.exists(), f"Missing processed dataset: {in_csv}"

df = pd.read_csv(in_csv)
df.shape, df.head(3)

((4000, 6),
   region_id chrom      start        end  \
 0    prom_0     4   88592334   88592533   
 1    prom_1     5  120078977  120079176   
 2    prom_2     8    9371095    9371294   
 
                                             sequence  label  
 0  TGCCCGCGGACCTTGCCGCCCCGCCTCCAGCCCGTGCCACGGCGGC...      1  
 1  AGAAAGAAGGAGAAAAAACGGCTCAAAGAAGAGTTGATGGCTGGGA...      1  
 2  TATGGAAAGCTTCACAAATTTGCACGGCATCTTTGTGCAGGGCCGT...      1  )

## Basic sanity checks

We validate:
- required columns exist
- no missing sequences/labels
- fixed sequence length == L
- characters are only A/C/G/T/N

In [1]:
required = ["region_id", "sequence", "label"]
missing = [c for c in required if c not in df.columns]
assert not missing, f"Missing columns: {missing}"

assert df["sequence"].isna().sum() == 0
assert df["label"].isna().sum() == 0

seq_len = df["sequence"].astype(str).str.len()
n_bad_len = int((seq_len != L).sum())

allowed = set("ACGTN")
def has_invalid(seq: str) -> bool:
    s = str(seq).upper()
    return any(ch not in allowed for ch in s)

invalid_flag = df["sequence"].astype(str).apply(has_invalid)
n_invalid = int(invalid_flag.sum())

n_bad_len, n_invalid, df["label"].value_counts()

NameError: name 'df' is not defined

## Feature functions

All functions are deterministic, fast, and work on fixed-length sequences.
We treat N as "unknown" and either:
- ignore it (entropy, skews denominators when needed)
- count it explicitly (N fraction)

In [3]:
VALID_BASES = set("ACGT")
ALL_BASES = set("ACGTN")

DINUCS = [a+b for a in "ACGT" for b in "ACGT"]                      # 16
TRINUCS = [a+b+c for a in "ACGT" for b in "ACGT" for c in "ACGT"]   # 64

def base_counts(seq: str):
    s = str(seq).upper()
    counts = {b: s.count(b) for b in "ACGTN"}
    return counts

def safe_div(num: float, den: float) -> float:
    return float(num) / float(den) if den != 0 else 0.0

def gc_content(seq: str) -> float:
    c = base_counts(seq)
    return safe_div(c["G"] + c["C"], len(seq))

def at_content(seq: str) -> float:
    c = base_counts(seq)
    return safe_div(c["A"] + c["T"], len(seq))

def gc_skew(seq: str) -> float:
    c = base_counts(seq)
    den = c["G"] + c["C"]
    return safe_div(c["G"] - c["C"], den)

def at_skew(seq: str) -> float:
    c = base_counts(seq)
    den = c["A"] + c["T"]
    return safe_div(c["A"] - c["T"], den)

def cpg_density(seq: str) -> float:
    s = str(seq).upper()
    return safe_div(s.count("CG"), max(len(s) - 1, 1))

def cpg_observed_expected(seq: str) -> float:
    s = str(seq).upper()
    c = base_counts(s)
    obs = s.count("CG")
    exp = safe_div(c["C"] * c["G"], max(len(s) - 1, 1))
    return safe_div(obs, exp)

def shannon_entropy_acgt(seq: str) -> float:
    """
    Shannon entropy over A/C/G/T only. Ignores N.
    Returns in bits.
    """
    s = str(seq).upper()
    counts = {b: s.count(b) for b in "ACGT"}
    total = sum(counts.values())
    if total == 0:
        return 0.0
    ent = 0.0
    for b in "ACGT":
        p = counts[b] / total
        if p > 0:
            ent -= p * math.log2(p)
    return float(ent)

def max_homopolymer_run(seq: str) -> int:
    """
    Max run length over A/C/G/T. Treats N as a break.
    """
    s = str(seq).upper()
    best = 0
    cur_char = None
    cur_run = 0
    for ch in s:
        if ch not in "ACGT":
            cur_char = None
            cur_run = 0
            continue
        if ch == cur_char:
            cur_run += 1
        else:
            cur_char = ch
            cur_run = 1
        best = max(best, cur_run)
    return int(best)

def kmer_freqs(seq: str, kmers: list[str]) -> dict:
    """
    Frequency per k-mer, normalized by number of k-mer slots.
    For seq length L and k:
      slots = L-k+1
    Any k-mer containing N won't match ACGT-only kmers -> effectively ignored.
    """
    s = str(seq).upper()
    k = len(kmers[0])
    slots = max(len(s) - k + 1, 1)
    out = {}
    for kmer in kmers:
        out[f"kmer_{k}_{kmer}"] = safe_div(s.count(kmer), slots)
    return out

def aa_tt_ta_fraction(seq: str) -> float:
    s = str(seq).upper()
    slots = max(len(s) - 1, 1)
    count = s.count("AA") + s.count("TT") + s.count("TA")
    return safe_div(count, slots)

def cpg_rich_flag(gc: float, cpg_oe: float) -> int:
    """
    Simple CpG-rich proxy flag (prototype).
    Thresholds are heuristic but common-ish:
      GC > 0.50 and CpG O/E > 0.60
    """
    return int((gc > 0.50) and (cpg_oe > 0.60))

## Build feature table

We compute features for each region and produce:

- `X`: baseline feature matrix
- `y`: labels
- plus metadata columns (`region_id`, optional chrom/start/end if present)

This is the feature table used by the modeling notebook.

In [4]:
def featurize_one(seq: str) -> dict:
    c = base_counts(seq)
    Lseq = len(str(seq))
    frac = {f"frac_{b}": safe_div(c[b], Lseq) for b in "ACGTN"}
    
    gc = gc_content(seq)
    at = at_content(seq)
    gcs = gc_skew(seq)
    ats = at_skew(seq)
    cpg_d = cpg_density(seq)
    cpg_oe = cpg_observed_expected(seq)
    ent = shannon_entropy_acgt(seq)
    hom = max_homopolymer_run(seq)
    per = aa_tt_ta_fraction(seq)
    flag_cpg = cpg_rich_flag(gc, cpg_oe)

    feats = {}
    feats.update(frac)
    feats["gc_content"] = gc
    feats["at_content"] = at
    feats["gc_skew"] = gcs
    feats["at_skew"] = ats
    feats["cpg_density"] = cpg_d
    feats["cpg_oe"] = cpg_oe
    feats["entropy_acgt"] = ent
    feats["max_homopolymer_run"] = hom
    feats["aa_tt_ta_fraction"] = per
    feats["is_cpg_rich"] = flag_cpg
    
    feats.update(kmer_freqs(seq, DINUCS))
    feats.update(kmer_freqs(seq, TRINUCS))
    return feats

# Build features
feat_rows = []
for seq in df["sequence"].astype(str).tolist():
    feat_rows.append(featurize_one(seq))

X = pd.DataFrame(feat_rows)
X.shape, X.head(2)

((4000, 95),
    frac_A  frac_C  frac_G  frac_T  frac_N  gc_content  at_content   gc_skew  \
 0   0.160   0.385   0.315    0.14     0.0       0.700       0.300 -0.100000   
 1   0.385   0.140   0.235    0.24     0.0       0.375       0.625  0.253333   
 
     at_skew  cpg_density  ...  kmer_3_TCG  kmer_3_TCT  kmer_3_TGA  kmer_3_TGC  \
 0  0.066667     0.125628  ...         0.0    0.000000    0.005051    0.025253   
 1  0.232000     0.010050  ...         0.0    0.020202    0.020202    0.005051   
 
    kmer_3_TGG  kmer_3_TGT  kmer_3_TTA  kmer_3_TTC  kmer_3_TTG  kmer_3_TTT  
 0    0.010101    0.005051    0.010101    0.015152    0.025253    0.000000  
 1    0.035354    0.005051    0.020202    0.010101    0.015152    0.010101  
 
 [2 rows x 95 columns])

## Assemble final feature dataset and validate

We create `df_feat` with:
- region_id, label
- baseline feature columns

We also check:
- no missing feature values
- expected feature count

In [5]:
meta_cols = [c for c in ["region_id", "chrom", "start", "end"] if c in df.columns]
df_feat = pd.concat([df[meta_cols + ["label"]].reset_index(drop=True), X.reset_index(drop=True)], axis=1)

# checks
assert df_feat.isna().sum().sum() == 0, "Found missing values in feature table."
assert df_feat.shape[0] == df.shape[0]

# expected feature counts
expected = (
    5  # frac_A/C/G/T/N
    + 1 + 1  # gc_content, at_content
    + 1 + 1  # gc_skew, at_skew
    + 1 + 1  # cpg_density, cpg_oe
    + 1      # entropy
    + 1      # homopolymer
    + 1      # periodic proxy
    + 1      # is_cpg_rich
    + 16     # 2-mers
    + 64     # 3-mers
)

n_feat_cols = df_feat.shape[1] - (len(meta_cols) + 1)  # minus meta + label
expected, n_feat_cols

(95, 95)

## Save artifacts

We save:
- Parquet feature table (fast + stable types)
- CSV feature table (optional / human-readable)
- JSON summary (schema + counts)

In [6]:
out_parquet = PROCESSED / f"dna_features_baseline_len{L}_pos{N_POS}_neg{N_NEG}.parquet"
out_csv = PROCESSED / f"dna_features_baseline_len{L}_pos{N_POS}_neg{N_NEG}.csv"

df_feat.to_parquet(out_parquet, index=False)
df_feat.to_csv(out_csv, index=False)

feature_cols = [c for c in df_feat.columns if c not in (meta_cols + ["label"])]

summary = {
    "input_dataset": str(in_csv),
    "output_parquet": str(out_parquet),
    "output_csv": str(out_csv),
    "n_rows": int(df_feat.shape[0]),
    "n_pos": int((df_feat["label"] == 1).sum()),
    "n_neg": int((df_feat["label"] == 0).sum()),
    "seq_length_bp": int(L),
    "n_features": int(len(feature_cols)),
    "feature_columns": feature_cols,
    "seed": int(SEED),
    "notes": {
        "kmer_2": "ACGT dinucleotide frequencies (16)",
        "kmer_3": "ACGT trinucleotide frequencies (64)",
        "entropy": "Shannon entropy over A/C/G/T ignoring N",
        "cpg": "CpG density + CpG observed/expected",
        "skews": "GC skew and AT skew",
        "periodicity_proxy": "AA/TT/TA dinucleotide fraction",
        "cpg_rich_flag": "GC>0.50 and CpG O/E>0.60 (prototype heuristic)",
    }
}

summary_path = REPORTS / "dna_baseline_features_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

summary_path

PosixPath('/Users/saturnine/Projects/bio-seq-lm-capstone/reports/dna_baseline_features_summary.json')

## Quick preview

This is just a quick peek at feature distributions to confirm nothing is wildly off.
Full EDA remains in the dedicated EDA notebook.

In [7]:
df_feat[["label", "gc_content", "cpg_density", "cpg_oe", "entropy_acgt", "max_homopolymer_run", "is_cpg_rich"]].describe(include="all")

,label,gc_content,cpg_density,cpg_oe,entropy_acgt,max_homopolymer_run,is_cpg_rich
count,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000
mean,0.500000,0.454612,0.023634,0.309251,1.852613,5.447000,0.134250
std,0.500063,0.151931,0.034014,0.290486,0.379565,3.134429,0.340963
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.375000,0.005025,0.099165,1.893065,4.000000,0.000000
50%,0.500000,0.450000,0.010050,0.235643,1.947130,5.000000,0.000000
75%,1.000000,0.540000,0.025126,0.444024,1.975835,6.000000,0.000000
max,1.000000,0.875000,0.231156,2.132143,2.000000,48.000000,1.000000
